In [ ]:
import gradio as gr
import torch
from ultralytics import YOLO

model_path = "runs/detect/train6/weights/best.pt"

model = YOLO(model_path)


device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
print(f"Using {device}")


def predict(image):
    result = model(image, max_det=500)[0]

    result_image = result.plot(labels=False)

    total_colonies = len(result.boxes.cls)

    return result_image, total_colonies


demo = gr.Interface(
    fn=predict,
    inputs=["image"],
    examples=[
        ["../data/unlabeled/default_mmrsbpb_default_19012024135839.jpg"],
        ["../data/unlabeled/default_mmrsbpb_default_19012024140321.jpg"],
        ["../data/unlabeled/default_mmrsbpb_default_19012024140822.jpg"],
        ["../data/unlabeled/default_mmrsbpb_mMRS-BPB_06122023155324.jpg"],
    ],
    outputs=[
        gr.Image(label="Segmented image"),
        gr.Number(label="Number of colonies detected"),
    ],
    title="Colony Detection",
    allow_flagging="never",
    live=True,
)

demo.launch(inline=False, server_name="0.0.0.0")